## 1. 5×5 matrix, 3×3 kernel, and one convolution

In [1]:
import numpy as np

# Create a reproducible random 5x5 matrix
np.random.seed(42)
input_matrix = np.random.randint(0, 10, (5, 5))

# Define a 3x3 kernel
kernel = np.array([
    [1,  0, -1],
    [1,  0, -1],
    [1,  0, -1]
])

print("Input Matrix:")
print(input_matrix)

print("\nKernel:")
print(kernel)

# One convolution operation: top-left 3x3 region
region = input_matrix[0:3, 0:3]

activation = np.sum(region * kernel)

print("\nSelected 3x3 Region:")
print(region)

print("\nActivation value:")
print(activation)

# Complete activation map, no padding, stride = 1
output = np.zeros((3, 3))

for i in range(3):
    for j in range(3):
        region = input_matrix[i:i+3, j:j+3]
        output[i, j] = np.sum(region * kernel)

print("\nActivation Map:")
print(output)

Input Matrix:
[[6 3 7 4 6]
 [9 2 6 7 4]
 [3 7 7 2 5]
 [4 1 7 5 1]
 [4 0 9 5 8]]

Kernel:
[[ 1  0 -1]
 [ 1  0 -1]
 [ 1  0 -1]]

Selected 3x3 Region:
[[6 3 7]
 [9 2 6]
 [3 7 7]]

Activation value:
-2

Activation Map:
[[ -2.  -1.   5.]
 [ -4.  -4.  10.]
 [-12.  -4.   9.]]


## 2. Convolution with a given stride

In [2]:
import numpy as np

def apply_stride_convolution(input_matrix, kernel, stride):
    input_rows, input_cols = input_matrix.shape
    kernel_rows, kernel_cols = kernel.shape

    # Calculate output dimensions
    output_rows = (input_rows - kernel_rows) // stride + 1
    output_cols = (input_cols - kernel_cols) // stride + 1

    output = np.zeros((output_rows, output_cols))

    # Slide kernel using the specified stride
    for i in range(output_rows):
        for j in range(output_cols):

            row_start = i * stride
            col_start = j * stride

            region = input_matrix[
                row_start:row_start + kernel_rows,
                col_start:col_start + kernel_cols
            ]

            output[i, j] = np.sum(region * kernel)

    return output


# 6x6 input matrix
input_matrix = np.array([
    [1, 2, 3, 4, 5, 6],
    [7, 8, 9, 1, 2, 3],
    [4, 5, 6, 7, 8, 9],
    [1, 2, 3, 4, 5, 6],
    [7, 8, 9, 1, 2, 3],
    [4, 5, 6, 7, 8, 9]
])

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

stride = 2

result = apply_stride_convolution(input_matrix, kernel, stride)

print("Activation Map:")
print(result)

Activation Map:
[[-6.  3.]
 [-6.  3.]]


## 3. Add zero-padding of 1

In [3]:
import numpy as np

def convolution_with_padding(input_matrix, kernel, padding=1, stride=1):

    # Add zero-padding
    padded_matrix = np.pad(
        input_matrix,
        ((padding, padding), (padding, padding)),
        mode='constant',
        constant_values=0
    )

    kernel_rows, kernel_cols = kernel.shape
    input_rows, input_cols = padded_matrix.shape

    output_rows = (input_rows - kernel_rows) // stride + 1
    output_cols = (input_cols - kernel_cols) // stride + 1

    output = np.zeros((output_rows, output_cols))

    for i in range(output_rows):
        for j in range(output_cols):

            row = i * stride
            col = j * stride

            region = padded_matrix[
                row:row + kernel_rows,
                col:col + kernel_cols
            ]

            output[i, j] = np.sum(region * kernel)

    return padded_matrix, output


# Same 5x5 matrix from Task 1
input_matrix = np.array([
    [6, 3, 7, 4, 6],
    [9, 2, 6, 7, 4],
    [3, 7, 7, 2, 5],
    [4, 1, 7, 5, 1],
    [4, 0, 9, 5, 8]
])

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

padded, activation_map = convolution_with_padding(
    input_matrix, kernel, padding=1, stride=1
)

print("Padded Matrix:")
print(padded)

print("\nActivation Map with Padding:")
print(activation_map)

Padded Matrix:
[[0 0 0 0 0 0 0]
 [0 6 3 7 4 6 0]
 [0 9 2 6 7 4 0]
 [0 3 7 7 2 5 0]
 [0 4 1 7 5 1 0]
 [0 4 0 9 5 8 0]
 [0 0 0 0 0 0 0]]

Activation Map with Padding:
[[ -5.   2.  -6.   3.  11.]
 [-12.  -2.  -1.   5.  13.]
 [-10.  -4.  -4.  10.  14.]
 [ -8. -12.  -4.   9.  12.]
 [ -1.  -8.  -9.   7.  10.]]


## 4. Edge detection vs. blur

In [4]:
import numpy as np

input_matrix = np.array([
    [6, 3, 7, 4, 6],
    [9, 2, 6, 7, 4],
    [3, 7, 7, 2, 5],
    [4, 1, 7, 5, 1],
    [4, 0, 9, 5, 8]
])

edge_kernel = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0]
])

blur_kernel = np.ones((3, 3)) / 9


def convolution(input_matrix, kernel):
    rows, cols = input_matrix.shape
    k_rows, k_cols = kernel.shape

    output = np.zeros(
        (rows - k_rows + 1, cols - k_cols + 1)
    )

    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            region = input_matrix[
                i:i+k_rows,
                j:j+k_cols
            ]

            output[i, j] = np.sum(region * kernel)

    return output


edge_output = convolution(input_matrix, edge_kernel)
blur_output = convolution(input_matrix, blur_kernel)

print("Edge Detection          Blur")
print("-------------------------------------")

for edge_row, blur_row in zip(edge_output, blur_output):
    print(
        " ".join(f"{x:6.1f}" for x in edge_row),
        "    ",
        " ".join(f"{x:6.1f}" for x in blur_row)
    )

Edge Detection          Blur
-------------------------------------
  17.0   -1.0  -12.0         5.6    5.0    5.3
 -15.0   -6.0   16.0         5.1    4.9    4.9
  14.0   -6.0   -5.0         4.7    4.8    5.4


## 5. Instagram photo → grayscale → 8×8 → custom filter

In [5]:
from PIL import Image
import numpy as np


def convolution(input_matrix, kernel):
    rows, cols = input_matrix.shape
    k_rows, k_cols = kernel.shape

    output = np.zeros(
        (rows - k_rows + 1, cols - k_cols + 1)
    )

    for i in range(output.shape[0]):
        for j in range(output.shape[1]):

            region = input_matrix[
                i:i+k_rows,
                j:j+k_cols
            ]

            output[i, j] = np.sum(region * kernel)

    return output


# Load Instagram photo
image = Image.open("instagram_photo.jfif")

# Convert to grayscale
gray_image = image.convert("L")

# Resize to 8x8
small_image = gray_image.resize((8, 8))

# Convert to NumPy array
input_array = np.array(small_image, dtype=float)

# Custom vertical-edge filter
custom_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

# Apply convolution
output_array = convolution(input_array, custom_filter)

print("8x8 Grayscale Input:")
print(np.round(input_array, 1))

print("\n6x6 Convolution Output:")
print(np.round(output_array, 1))


8x8 Grayscale Input:
[[155. 165. 135. 176. 183. 183. 181. 177.]
 [174. 173. 104. 187. 199. 200. 197. 191.]
 [187. 179. 108. 199. 195. 182. 186. 204.]
 [155. 196. 183. 125.  94.  70.  78. 124.]
 [111. 211. 203.  72.  58.  70.  56.  34.]
 [129. 183. 191. 105.  76.  77.  69.  62.]
 [ 81.  98. 184. 142.  94.  95.  91.  80.]
 [ 70. 114. 184. 208. 110.  72.  74.  76.]]

6x6 Convolution Output:
[[-169.   45.  230.    3.  -13.    7.]
 [-121.  -37.   93.  -59.  -27.   67.]
 [  41. -190. -147.  -74.  -27.   40.]
 [ 182. -288. -349.  -85.  -25.    3.]
 [ 257. -173. -350.  -77.  -12.  -66.]
 [ 279.   60. -279. -211.  -46.  -26.]]
